# Chest X-Ray Classification Pipeline
**4-Class ROI Dataset | VGG16 → Bridge → Swin-Base | 384×384**

This notebook runs the full pipeline:
1. Install dependencies
2. Clone the code repository
3. Download the 4-class ROI dataset from HuggingFace
4. Train the model (`C_train.py`)
5. Evaluate on the test set (`D_test.py`)
6. Plot training curves (`E_visualize.py`)
7. Generate Grad-CAM visualizations (`F_gradcam.py`)

## Step 1 — Install Dependencies

In [ ]:
!pip install -q torch torchvision timm scikit-learn pandas numpy matplotlib seaborn tqdm hf
print('Dependencies installed.')

## Step 2 — Clone the Code Repository

In [ ]:
import os

REPO_URL  = 'https://github.com/amhyou/chest-x-ray-data-preparation.git'
REPO_DIR  = 'chest-x-ray-data-preparation'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
else:
    print('Repo already cloned. Pulling latest...')
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

## Step 3 — Download 4-Class ROI Dataset from HuggingFace

Dataset: [loomolivet/chest-x-ray-roi](https://huggingface.co/datasets/loomolivet/chest-x-ray-roi)  
File: `roi_4cls_x384.zip`

In [ ]:
from huggingface_hub import hf_hub_download
import zipfile

HF_REPO   = 'loomolivet/chest-x-ray-roi'
HF_FILE   = 'roi_4cls_x384.zip'
ZIP_PATH  = HF_FILE

# Download
print(f'Downloading {HF_FILE} from HuggingFace...')
local_zip = hf_hub_download(
    repo_id=HF_REPO,
    filename=HF_FILE,
    repo_type='dataset',
    local_dir='.'
)
print(f'Downloaded to: {local_zip}')

# Extract
print('Extracting...')
with zipfile.ZipFile(local_zip, 'r') as z:
    z.extractall('.')
print('Extraction complete.')

# Verify expected structure
expected_dirs = ['data_roi_4class', 'metadata']
for d in expected_dirs:
    status = '✓' if os.path.exists(d) else '✗ MISSING'
    print(f'  {status}  {d}/')

In [ ]:
# Quick sanity check
import pandas as pd

csv_path = 'metadata/DATA_ROI_4CLASS.csv'
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f'Metadata loaded: {len(df)} rows, {df["Image_ID"].nunique()} unique images')
    print(f'Columns: {list(df.columns)}')
    print(f'Class counts:')
    for cls in ['Atelectasis', 'Cardiomegaly', 'Effusion', 'Normal']:
        if cls in df.columns:
            print(f'  {cls}: {df[cls].sum()}')
else:
    print(f'ERROR: {csv_path} not found. Check zip structure.')

## Step 4 — Verify Config (4-class)

In [ ]:
import importlib
import config

# Ensure we are in 4-class mode
assert config.NUM_CLASSES == 4, 'Set NUM_CLASSES = 4 in config.py!'

print('Config:')
print(f'  NUM_CLASSES   = {config.NUM_CLASSES}')
print(f'  ROI_IMAGE_DIR = {config.ROI_IMAGE_DIR}  exists={os.path.exists(config.ROI_IMAGE_DIR)}')
print(f'  METADATA_PATH = {config.METADATA_PATH}  exists={os.path.exists(config.METADATA_PATH)}')
print(f'  CHECKPOINT_DIR= {config.CHECKPOINT_DIR}')
print(f'  IMG_SIZE      = {config.IMG_SIZE}')
print(f'  BATCH_SIZE    = {config.BATCH_SIZE} × {config.ACCUMULATION_STEPS} = {config.BATCH_SIZE*config.ACCUMULATION_STEPS} effective')

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\nDevice: {device}')
if device == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Step 5 — Train

Training runs 3 phases automatically:
- **Warmup** (5 epochs, LR=1e-3): Only head is trained
- **Phase 1** (20 epochs, LR=1e-4): Swin stages unfrozen
- **Phase 2** (15 epochs, LR=1e-5): Full fine-tune with discriminative LR

Best model (by val AUC) is saved to `weights/best_model_fold0.pth`.

In [ ]:
!python C_train.py --fold 0

## Step 6 — Plot Training Curves

In [ ]:
!python E_visualize.py

In [ ]:
# Display the plots inline
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from glob import glob

plot_files = sorted(glob('results/*.png'))
print(f'Found {len(plot_files)} plots:')
for p in plot_files:
    print(f'  {p}')
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.imshow(mpimg.imread(p))
    ax.axis('off')
    ax.set_title(os.path.basename(p))
    plt.tight_layout()
    plt.show()

## Step 7 — Evaluate on Test Set

In [ ]:
!python D_test.py

In [ ]:
# (Optional) Optimize per-class thresholds, then re-run D_test.py
!python threshold_optimizer.py
print('\nRe-running test with optimal thresholds...')
!python D_test.py

In [ ]:
# Display test result plots
test_plots = sorted(glob('results/*.png'))
for p in test_plots:
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.imshow(mpimg.imread(p))
    ax.axis('off')
    ax.set_title(os.path.basename(p))
    plt.tight_layout()
    plt.show()

## Step 8 — Grad-CAM Visualizations

> **Note:** Heatmaps are computed on the ROI images (what the model trained on) and overlaid on the original raw X-rays. If you haven't downloaded the raw dataset, overlays will fall back to ROI images automatically.

In [ ]:
!python F_gradcam.py

In [ ]:
# Display Grad-CAM results per class
from glob import glob

gradcam_files = sorted(glob('results/gradcam/**/*.png', recursive=True))
print(f'Generated {len(gradcam_files)} Grad-CAM images')

for p in gradcam_files[:10]:   # show first 10
    fig, ax = plt.subplots(figsize=(18, 5))
    ax.imshow(mpimg.imread(p))
    ax.axis('off')
    ax.set_title(os.path.basename(p), fontsize=10)
    plt.tight_layout()
    plt.show()

## Step 9 — Save Results

Zip the results and weights for download or storage.

In [ ]:
import shutil

shutil.make_archive('run_results', 'zip', '.', 'results')
shutil.make_archive('run_weights', 'zip', '.', 'weights')
print('Saved: run_results.zip and run_weights.zip')